In [ ]:
import pandas as pd
import numpy as np

# версия pandas - полезно фиксировать в отчётах/дипломах
print(pd.__version__)

### Series

In [ ]:
# Создание Series из списка (индекс по умолчанию - 0, 1, 2, ...)
s = pd.Series([10, 20, 30, 40])
print(s)
print(type(s))


In [ ]:
# Создание Series с содержательным индексом
gdp = pd.Series(
    [4.07, 2.78, 2.10, 1.77],
    index=["DE", "FR", "IT", "ES"],
    name="ВВП (трлн USD)"
)
gdp

In [ ]:
# Доступ по метке
print(gdp["DE"])          # одно значение
print(gdp[["DE", "IT"]])  # несколько значений -> снова Series


In [ ]:
inflation = pd.Series({"DE": 2.1, "FR": 2.5, "IT": 3.0, "ES": 3.5}, name="Инфляция, %")

real_gdp = gdp - (gdp * inflation / 100)
real_gdp.name = "Реальный ВВП (оценка)"
real_gdp

In [ ]:
# Что будет, если индексы не совпадают полностью?
inflation_partial = pd.Series({"DE": 2.1, "FR": 2.5, "PL": 4.0})  # добавили PL, убрали IT и ES

result = gdp + inflation_partial
result  # там, где нет пары -> NaN.


### DataFrame
`DataFrame` можно создать множеством способов. Разберём самые частые на практике.

In [ ]:
# Из словаря списков (самый частый способ вручную)
df = pd.DataFrame({
    "страна":    ["Германия", "Франция", "Италия", "Испания"],
    "ввп":       [4.07, 2.78, 2.10, 1.77],
    "инфляция":  [2.1, 2.5, 3.0, 3.5],
    "население": [83.2, 67.8, 59.6, 47.4],
})
df


In [ ]:
# Из списка словарей - типичный формат данных из API / JSON
records = [
    {"id": 1, "сумма": 250.0, "категория": "еда"},
    {"id": 2, "сумма": 1200.0, "категория": "электроника"},
    {"id": 3, "сумма": 89.5, "категория": "еда"},
]
df2 = pd.DataFrame(records)
df2


In [ ]:
# Из массива NumPy
arr = np.random.randn(5, 3)
df3 = pd.DataFrame(arr, columns=["A", "B", "C"])
df3


In [ ]:
# Из другого DataFrame (выбор подмножества столбцов)
df4 = pd.DataFrame(df[["страна", "ввп"]])
df4


### Базовые атрибуты и методы обзора

Метаданные датафрейма


In [ ]:
print("Размер (строки, столбцы):", df.shape)
print("\nТипы столбцов:\n", df.dtypes)
print("\nИмена столбцов:", list(df.columns))
print("\nИндекс:", df.index)

In [ ]:
df.info()

In [ ]:
df.describe() # статистика только по числовым столбцам

In [ ]:
display(df.head(2))  # первые 2 строки
display(df.tail(2))  # последние 2 строки
print("Количество строк:", len(df))

### Базовые операции со столбцами

In [ ]:
# Выбор одного столбца -> Series
print(type(df["ввп"]))
df["ввп"]

In [ ]:
# Выбор нескольких столбцов -> DataFrame
df[["страна", "ввп"]]

In [ ]:
# Добавление вычисляемого столбца: ВВП на душу населения (USD)
df["ввп_на_душу"] = df["ввп"] * 1e12 / (df["население"] * 1e6)
df

In [ ]:
# Удаление столбца
df_short = df.drop(columns=["ввп_на_душу"])
df_short

In [ ]:
# Переименование столбцов
df_en = df.rename(columns={"ввп": "gdp_trn", "инфляция": "inflation_pct"})
df_en.head()

In [ ]:
# Сортировка по значению столбца
df.sort_values("ввп_на_душу", ascending=False)

### Ввод/вывод (I/O): чтение и запись данных

Все функции чтения имеют префикс `read_*`, все методы записи - префикс `to_*`.
Это единый паттерн для всей библиотеки.

| Формат  | Чтение              | Запись            | Комментарий |
|---------|---------------------|-------------------|-------------|
| CSV     | `pd.read_csv()`     | `df.to_csv()`     | универсальный, но медленный и без типов |
| Excel   | `pd.read_excel()`   | `df.to_excel()`   | удобно для отчётов, медленно на больших данных |
| JSON    | `pd.read_json()`    | `df.to_json()`    | поддерживает вложенность |
| Parquet | `pd.read_parquet()` | `df.to_parquet()` | быстро, компактно, сохраняет типы - предпочтительно для промежуточного хранения |
| SQL     | `pd.read_sql()`     | `df.to_sql()`     | нужно подключение к БД |


In [ ]:
# Создаём учебный CSV-файл "с нуля", чтобы дальше его же и прочитать
demo_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "дата": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19"],
    "сумма": [250.0, 1200.5, np.nan, 89.5, 3000.0],
    "категория": ["еда", "электроника", "еда", "транспорт", "электроника"],
})
demo_df.to_csv("data/test.csv", index=False, encoding="utf-8")
print("Файл test.csv создан")
demo_df

In [ ]:
# Базовое чтение
df_csv = pd.read_csv("data/test.csv")
df_csv.dtypes

In [ ]:
# Чтение: явные параметры (хорошая практика на реальных данных)
df_csv = pd.read_csv(
    "data/transactions.csv",
    sep=",",                     # разделитель (в европейских выгрузках часто ";")
    encoding="utf-8",
    parse_dates=["дата"],        # сразу распознать как datetime
    dtype={"id": "int64"},       # явно задать типы, где это важно
    na_values=["N/A", "-", ""],  # что считать пропуском
)
df_csv.dtypes

*Частая проблема автоопределения типов*: pandas сам угадывает тип столбца,
и иногда неправильно - например, `"01"` может быть прочитан как строка вместо
целого числа, а `"2024-01"` - как строка вместо даты. *Всегда проверяйте
`df.dtypes` сразу после чтения файла.**


In [ ]:
# Чтение большого файла по частям (chunksize) - актуально, когда файл не помещается в RAM
results = []
for chunk in pd.read_csv("data/transactions.csv", chunksize=2):
    filtered = chunk[chunk["сумма"] > 500]
    results.append(filtered)

df_big = pd.concat(results, ignore_index=True)
df_big

### JSON


In [ ]:
import json

# Создаём учебный JSON с вложенной структурой (как часто отдают API)
nested_data = [
    {
        "client_id": "C001",
        "transactions": [
            {"сумма": 250.0, "категория": "еда"},
            {"сумма": 1200.5, "категория": "электроника"},
        ],
    },
    {
        "client_id": "C002",
        "transactions": [
            {"сумма": 89.5, "категория": "еда"},
        ],
    },
]
with open("data/nested.json", "w", encoding="utf-8") as f:
    json.dump(nested_data, f, ensure_ascii=False)

# Простое чтение (для плоского JSON подошёл бы pd.read_json)
with open("data/nested.json", encoding="utf-8") as f:
    data = json.load(f)

# Разворачиваем вложенную структуру в плоскую таблицу
df_json = pd.json_normalize(data, record_path="transactions", meta=["client_id"])
df_json

### Parquet 


In [ ]:
try:
    df_csv.to_parquet("data/output.parquet", index=False)
    df_parquet = pd.read_parquet("data/output.parquet")
    print("Parquet прочитан успешно:")
    display(df_parquet)
except ImportError as e:
    print("Для работы с Parquet установите пакет pyarrow: pip install pyarrow")
    print("Ошибка:", e)

### Настройки отображения и метаданные


In [ ]:
pd.options.display.max_rows = 100
pd.options.display.max_columns = None
pd.set_option("display.float_format", "{:.2f}".format)

df_csv.info()

In [ ]:
df_csv.memory_usage(deep=True)

### Объединение таблиц: `pd.concat`


In [ ]:
df_a = pd.DataFrame({"страна": ["DE", "FR"], "ввп": [4.07, 2.78]})
df_b = pd.DataFrame({"страна": ["IT", "ES"], "ввп": [2.10, 1.77]})

# Вертикально (одна под другой)
vertical = pd.concat([df_a, df_b], ignore_index=True)
display(vertical)

# Горизонтально (рядом, по столбцам)
df_c = pd.DataFrame({"инфляция": [2.1, 2.5]})
horizontal = pd.concat([df_a, df_c], axis=1)
display(horizontal)


### Индексация и фильтрация


In [ ]:
df_idx = pd.DataFrame({
    "страна":  ["Германия", "Франция", "Италия", "Испания"],
    "ввп":     [4.07, 2.78, 2.10, 1.77],
    "рейтинг": [1, 2, 3, 4],
}, index=["DE", "FR", "IT", "ES"])
df_idx

In [ ]:
# .loc - доступ ПО МЕТКЕ (обратите внимание: правая граница ВКЛЮЧАЕТСЯ!)
print(df_idx.loc["DE"])
print()
print(df_idx.loc["DE", "ввп"])
print()
print(df_idx.loc["DE":"IT", "ввп"])   # срез включает и "IT"

In [ ]:
# .iloc - доступ ПО ПОЗИЦИИ (как обычные Python-срезы: правая граница НЕ включается)
print(df_idx.iloc[0])
print()
print(df_idx.iloc[0, 1])
print()
print(df_idx.iloc[0:2])   # строки 0 и 1, строка с индексом 2 НЕ включена

In [ ]:
# Типичная ошибка новичков:
try:
    df_idx.loc[0]  # ошибка: метки 0 в индексе нет (индекс строковый: DE, FR, ...)
except KeyError as e:
    print("KeyError:", e)

try:
    df_idx.iloc["DE"]  # ошибка: iloc принимает только позиции (числа)
except TypeError as e:
    print("TypeError:", e)

### Булевы маски

In [ ]:
df_mask = df_csv.copy()

# Простой фильтр
mask = df_mask["сумма"] > 500
df_expensive = df_mask[mask]
df_expensive

In [ ]:
# Несколько условий - скобки обязательны!
df_select = df_mask[(df_mask["сумма"] > 100) & (df_mask["категория"] == "еда")]
display(df_select)

df_or = df_mask[(df_mask["категория"] == "еда") | (df_mask["категория"] == "транспорт")]
display(df_or)

In [ ]:
# isin - проверка вхождения в список значений
categories = ["еда", "транспорт"]
df_needed = df_mask[df_mask["категория"].isin(categories)]
display(df_needed)

# ~ - инверсия маски (всё, что НЕ попало в список)
df_rest = df_mask[~df_mask["категория"].isin(categories)]
display(df_rest)

In [ ]:
# notna / isna - работа с пропусками
print("Строки без пропуска в 'сумма':")
display(df_mask[df_mask["сумма"].notna()])

print("Строки с пропуском в 'сумма':")
display(df_mask[df_mask["сумма"].isna()])

### `.query()` - более читаемый синтаксис фильтрации

In [ ]:
df_select_q = df_mask.query("сумма > 100 and категория == 'еда'")
df_select_q

In [ ]:
# С переменными из окружения - через @
threshold = 500
df_dorogie_q = df_mask.query("сумма > @threshold")
df_dorogie_q

### `.copy()` и `SettingWithCopyWarning`


In [ ]:
# Проблема: изменение среза может выдать предупреждение и работать непредсказуемо
df_food = df_mask[df_mask["категория"] == "еда"]
df_food["наценка"] = df_food["сумма"] * 0.1  # ⚠️ SettingWithCopyWarning
df_food

In [ ]:
# Решение: явно создаём копию перед изменением
df_food = df_mask[df_mask["категория"] == "еда"].copy()
df_food["наценка"] = df_food["сумма"] * 0.1  # теперь безопасно и предсказуемо
df_food

### Работа со столбцами

In [ ]:
df_apply = df_csv.copy()

# К одному столбцу (Series)
df_apply["сумма_с_ндс"] = df_apply["сумма"].apply(lambda x: x * 1.2)
df_apply

In [ ]:
def classify(amount):
    if pd.isna(amount):
        return "неизвестно"
    if amount < 100:
        return "малая"
    elif amount < 1000:
        return "средняя"
    return "крупная"

df_apply["размер"] = df_apply["сумма"].apply(classify)
df_apply

In [ ]:
# apply с axis=1 -- применяется к каждой строке целиком
df_apply["описание"] = df_apply.apply(
    lambda row: f"{row['категория']}: {row['сумма']:.2f} руб." if pd.notna(row["сумма"]) else "нет данных",
    axis=1
)
df_apply[["категория", "сумма", "описание"]]

### `.map()` - точечная замена значений

In [ ]:
cat_code = {"еда": 1, "транспорт": 2, "электроника": 3}
df_apply["кат_id"] = df_apply["категория"].map(cat_code)
df_apply[["категория", "кат_id"]]

### Строковые методы `.str`

In [ ]:
s_cat = pd.Series(["  Еда ", "ЭЛЕКТРОНИКА", "еда", "Транспорт-Такси"])

print(s_cat.str.lower().str.strip().tolist())
print(s_cat.str.contains("такси", case=False, na=False).tolist())
print(s_cat.str.startswith("Е").tolist())

In [ ]:
# Разбивка строки на несколько столбцов
names = pd.Series(["Иван Иванов", "Мария Петрова", "Сергей Сидоров"])
names_df = names.str.split(" ", expand=True)
names_df.columns = ["Имя", "Фамилия"]
names_df

In [ ]:
# Извлечение по регулярному выражению
articles = pd.Series(["AB1234-товар", "CD5678-услуга", "нет кода"])
codes = articles.str.extract(r"([A-Z]{2}\d{4})")
codes

### Переименование столбцов и приведение типов

In [ ]:
df_types = df_csv.rename(columns={
    "сумма": "amount",
    "категория": "category",
    "дата": "date",
})

# Привести все имена столбцов к единому стилю
df_types.columns = df_types.columns.str.lower().str.replace(" ", "_")

# Приведение типов
df_types["amount"] = df_types["amount"].astype(float)
df_types["category"] = df_types["category"].astype("category")  # экономит память
df_types["date"] = pd.to_datetime(df_types["date"])

df_types.dtypes

### Загрузка и очистка транзакционного датасета


In [ ]:
data = {
    "ID":        ["T001", "T002", "T003", "T004", "T005"],
    "Сумма":     ["250,00", "1 200.50", "N/A", "89.5", "3000"],
    "Категория": ["  Еда ", "ЭЛЕКТРОНИКА", "еда", "Транспорт", "электроника"],
    "Дата":      ["2024-01-15", "15/01/2024", "2024-01-16", "2024-01-17", "2024-01-18"],
    "Клиент":    ["Иван Иванов", "Мария Петрова", None, "Сергей Сидоров", "Анна Козлова"],
}
df_dirty = pd.DataFrame(data)

print("=== Исходные данные ===")
display(df_dirty)
print(df_dirty.dtypes)

In [ ]:
df_clean = df_dirty.copy()

# Шаг 1: очистка столбца "Сумма"
df_clean["Сумма"] = (
    df_clean["Сумма"]
    .str.replace(",", ".", regex=False)   # запятая -> точка (европейский десятичный разделитель)
    .str.replace(" ", "", regex=False)    # убрать разделители тысяч
    .replace("N/A", pd.NA)                # явный пропуск вместо текстовой заглушки
    .astype(float)
)
df_clean

In [ ]:
# Шаг 2: нормализация категорий (регистр + пробелы по краям)
df_clean["Категория"] = df_clean["Категория"].str.strip().str.lower()
df_clean

In [ ]:
# Шаг 3: разбор дат в разных форматах.
df_clean["Дата"] = pd.to_datetime(df_clean["Дата"], format="mixed", dayfirst=True, errors="coerce")
df_clean

In [ ]:
# Шаг 4: разбивка имени клиента на Имя/Фамилию
df_clean[["Имя", "Фамилия"]] = df_clean["Клиент"].str.split(" ", expand=True)
df_clean

In [ ]:
print("=== После очистки ===")
display(df_clean)
print(df_clean.dtypes)

In [ ]:
df_clean.groupby("Категория")["Сумма"].agg(["count", "sum", "mean"]).round(2)